In [ ]:
!pip install git+https://github.com/facebookresearch/fastMRI.git 

In [ ]:
import h5py
import numpy as np
import torch
import os
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
import fastmri
from fastmri.data.subsample import RandomMaskFunc
from fastmri import fft2c, ifft2c, complex_abs
from fastmri.data.transforms import to_tensor, complex_center_crop

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

# === Step 1: Point to your dataset folder ===
folder_path = r'D:\chrome_download\mridatasetdownload\M4RawV1.5_multicoil_train\multicoil_train' 
if not os.path.exists(folder_path):
    print("Folder does not exist!")
else:
    all_files = os.listdir(folder_path)
    print(f"Found {len(all_files)} files in folder.")
    print("First few files:", all_files[:5])
    h5_files = [f for f in all_files if f.endswith('.h5')]
    print("H5 files found:", h5_files)

file_name = h5_files[0]  # Change index to pick a different file
file_path = os.path.join(folder_path, file_name)
print(f"Opening file: {file_name}")

with h5py.File(file_path, 'r') as hf:
    print("Keys in file:", list(hf.keys()))
    volume_kspace = hf['kspace'][()]  # shape: (slices, coils, H, W)
    print("Volume shape:", volume_kspace.shape)

    slice_index = 10  # Can change to any index < num_slices
    slice_kspace = volume_kspace[slice_index]  # shape: (coils, H, W)
    print("Slice shape:", slice_kspace.shape)

# === Step 5: Show log-magnitude of k-space for up to 4 coils ===
def show_coils(data, coil_nums, cmap='gray'):
    plt.figure(figsize=(15, 5))
    for i, num in enumerate(coil_nums):
        plt.subplot(1, len(coil_nums), i + 1)
        plt.imshow(data[num], cmap=cmap)
        plt.title(f"Coil {num}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# === Step 6: Compute log-magnitude ===
log_mag = np.log(np.abs(slice_kspace) + 1e-9)  # Avoid log(0)

# === Step 7: Visualize for first 4 coils ===
num_coils = slice_kspace.shape[0]
coil_indices = list(range(min(4, num_coils)))  # [0, 1, 2, 3] or fewer
show_coils(log_mag, coil_indices)


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt



with h5py.File(file_path, 'r') as hf:
    print('Keys:', list(hf.keys()))
    print('Attrs:', dict(hf.attrs))
    
    # Load data
    volume_kspace = hf['kspace'][()]                # shape: (slices, coils, H, W)
    volume_img_rss = hf['reconstruction_rss'][()]   # shape: (slices, H, W)

    slice_idx = 10
    slice_kspace = volume_kspace[slice_idx]         # shape: (coils, H, W)
    slice_img_rss = volume_img_rss[slice_idx]       # shape: (H, W)

    print('K-space slice shape:', slice_kspace.shape)
    print('RSS image shape:', slice_img_rss.shape)

    # Visualize
    plt.figure(figsize=(6, 6))
    plt.imshow(slice_img_rss, cmap='gray')
    plt.title(f"Fully Sampled RSS Image (Slice {slice_idx})")
    plt.axis('off')
    plt.show()


In [ ]:

import h5py
import numpy as np
import torch
import os
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric, peak_signal_noise_ratio as psnr_metric
from fastmri.data.subsample import RandomMaskFunc
from fastmri.data import transforms as T
from fastmri import fft2c, ifft2c, complex_abs, rss
import bisect


class Args:
    folder_path = r'D:\chrome_download\mridatasetdownload\M4RawV1.5_multicoil_train\multicoil_train'
    center_fractions = [0.08]
    accelerations = [4]
    epoch_start = 0
    n_epochs = 50
    batch_size = 1
    lr = 1e-4
    b1, b2 = 0.9, 0.999
    iters = 2
    k_layers = 5
    i_layers = 5
    fm = 32
    use_stat_priors = True# True = stat-aware denoisers, False = plain CNN
    estimate_sigma = True # estimate noise sigma from k-space corners
    sigma_min = 0.005           # minimum sigma fallback
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

opt = Args()
print(f"Using device: {opt.device}")
os.makedirs('SavedModels_KIKI', exist_ok=True)
os.makedirs('Validation_KIKI', exist_ok=True)

# Dataset
if not os.path.exists(opt.folder_path):
    print(f"Error: Data folder not found at {opt.folder_path}")
    all_files = []
else:
    all_files = [os.path.join(opt.folder_path, f) for f in os.listdir(opt.folder_path) if f.endswith('.h5')]

if not all_files:
     print("Warning: No .h5 files found in the data folder. The program will exit if there's no data.")
else:
    print(f"Found {len(all_files)} files in the data folder.")

class FastMRIDataset(Dataset):
    def __init__(self, file_paths, mask_func):
        super().__init__()
        self.file_paths = file_paths
        self.mask_func = mask_func
        self.num_slices_per_file = []
        self.cumulative_slices = [0]
        for file_path in self.file_paths:
            try:
                with h5py.File(file_path, 'r') as hf:
                    num_slices = hf['kspace'].shape[0]
                    self.num_slices_per_file.append(num_slices)
                    self.cumulative_slices.append(self.cumulative_slices[-1] + num_slices)
            except (IOError, KeyError) as e:
                print(f"Warning: Could not read file {file_path}. Skipping. Error: {e}")

    def __len__(self):
        return self.cumulative_slices[-1]

    def __getitem__(self, idx):
        file_idx = bisect.bisect_right(self.cumulative_slices, idx) - 1
        local_slice_idx = idx - self.cumulative_slices[file_idx]
        with h5py.File(self.file_paths[file_idx], 'r') as hf:
            slice_np = hf['kspace'][local_slice_idx] # (coils, H, W)
        kspace_tensor = T.to_tensor(slice_np) # (coils, H, W, 2)
        masked_kspace, mask, _ = T.apply_mask(kspace_tensor, self.mask_func)
        img_fs = ifft2c(kspace_tensor)
        img_fs_rss = rss(complex_abs(img_fs), dim=0)
        return {
            'kspace_us': masked_kspace.permute(3,0,1,2),  # (2, coils, H, W)
            'img_fs': img_fs_rss.float(),
            'mask': mask.float(),
            'kspace_fs': kspace_tensor
        }

# FFT Utilities
def fft2c_torch(x):
    return fft2c(x)

def ifft2c_torch(x):
    return ifft2c(x)

def apply_IDC(k_rec, k_us, mask):
    return k_rec*(1-mask) + k_us*mask

# Noise Sigma Estimation 
def estimate_sigma_from_kspace(kspace_tensor, corner=0.05):
    with torch.no_grad():
        if kspace_tensor.dim() == 5:
            B,C,H,W,_ = kspace_tensor.shape
            kspace_tensor = kspace_tensor.view(B*C,H,W,2)
        else:
            C,H,W,_ = kspace_tensor.shape
            kspace_tensor = kspace_tensor.view(C,H,W,2)
        mag = torch.sqrt(kspace_tensor[...,0]**2 + kspace_tensor[...,1]**2)
        h_cut = int(H*corner)
        w_cut = int(W*corner)
        corners = torch.cat([
            mag[:,:h_cut,:w_cut].reshape(-1),
            mag[:,:h_cut,-w_cut:].reshape(-1),
            mag[:,-h_cut:,:w_cut].reshape(-1),
            mag[:,-h_cut:,-w_cut:].reshape(-1)
        ], dim=0)
        sigma = torch.std(corners).item()
    return sigma

# CNN Denoiser Base 
class CNN_Denoiser(nn.Module):
    def __init__(self, channels=32, num_layers=5):
        super().__init__()
        layers = [nn.Conv2d(2, channels, 3, 1, 1), nn.LeakyReLU(0.1,inplace=True)]
        for _ in range(num_layers-2):
            layers.extend([nn.Conv2d(channels, channels, 3,1,1), nn.LeakyReLU(0.1,inplace=True)])
        layers.append(nn.Conv2d(channels,2,1,1,0))
        self.net = nn.Sequential(*layers)
    def forward(self,x,sigma=None):
        return x + self.net(x)

#Stat-Aware Denoisers 
class GaussianStatDenoiser(nn.Module):
    def __init__(self, channels=32, num_layers=5):
        super().__init__()
        layers = [nn.Conv2d(2, channels, 3,1,1), nn.LeakyReLU(0.1,inplace=True)]
        for _ in range(num_layers-2):
            layers.extend([nn.Conv2d(channels, channels, 3,1,1), nn.LeakyReLU(0.1,inplace=True)])
        layers.append(nn.Conv2d(channels,2,1,1,0))
        self.base = nn.Sequential(*layers)
    def forward(self,x,sigma=0.05):
        sigma_t = torch.tensor(max(sigma,opt.sigma_min),dtype=x.dtype,device=x.device)
        residual = self.base(x)
        return x + residual * torch.exp(-0.5/(sigma_t**2))

class RicianStatDenoiser(nn.Module):
    def __init__(self, channels=32,num_layers=5):
        super().__init__()
        layers = [nn.Conv2d(2, channels,3,1,1), nn.LeakyReLU(0.1,inplace=True)]
        for _ in range(num_layers-2):
            layers.extend([nn.Conv2d(channels,channels,3,1,1), nn.LeakyReLU(0.1,inplace=True)])
        layers.append(nn.Conv2d(channels,2,1,1,0))
        self.base = nn.Sequential(*layers)
    def forward(self,x,sigma=0.05):
        sigma_t = torch.tensor(max(sigma,opt.sigma_min),dtype=x.dtype,device=x.device)
        residual = self.base(x)
        mag = torch.sqrt(x[:,0]**2 + x[:,1]**2 + 1e-8)
        bias = sigma_t**2 / (mag + 1e-8)
        bias = bias.unsqueeze(1).expand_as(x)
        return x + residual - bias

# Modified KIKI 
class KIKI(nn.Module):
    def __init__(self,opt):
        super().__init__()
        self.K_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(2,opt.fm,3,1,1), nn.LeakyReLU(0.1,True),
                *[nn.Sequential(nn.Conv2d(opt.fm,opt.fm,3,1,1),nn.LeakyReLU(0.1,True)) for _ in range(opt.k_layers-2)],
                nn.Conv2d(opt.fm,2,1,1,0)
            ) for _ in range(opt.iters+1)
        ])
        self.I_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(2,opt.fm,3,1,1), nn.LeakyReLU(0.1,True),
                *[nn.Sequential(nn.Conv2d(opt.fm,opt.fm,3,1,1),nn.LeakyReLU(0.1,True)) for _ in range(opt.i_layers-2)],
                nn.Conv2d(opt.fm,2,1,1,0)
            ) for _ in range(opt.iters)
        ])
        if opt.use_stat_priors:
            self.G_denoisers = nn.ModuleList([GaussianStatDenoiser(opt.fm,opt.k_layers) for _ in range(opt.iters)])
            self.R_denoisers = nn.ModuleList([RicianStatDenoiser(opt.fm,opt.i_layers) for _ in range(opt.iters)])
        else:
            self.G_denoisers = nn.ModuleList([CNN_Denoiser(opt.fm,opt.k_layers) for _ in range(opt.iters)])
            self.R_denoisers = nn.ModuleList([CNN_Denoiser(opt.fm,opt.i_layers) for _ in range(opt.iters)])
        self.iters = opt.iters
        self.sigma_min = opt.sigma_min

    def forward(self,k_us,mask,sigma=0.01):
        k_us = k_us.permute(0,2,3,4,1).contiguous()
        k_rec = k_us.clone()
        for i in range(self.iters):
            b,c,h,w,_ = k_rec.shape
            # K-space conv
            k_conv = k_rec.permute(0,1,4,2,3).reshape(b*c,2,h,w)
            k_conv = self.K_blocks[i](k_conv)
            k_conv = k_conv.view(b,c,2,h,w).permute(0,1,3,4,2)
            k_rec = k_rec + k_conv
            # Gaussian denoise 
            k_d = k_rec.permute(0,1,4,2,3).reshape(b*c,2,h,w)
            k_d = self.G_denoisers[i](k_d,sigma=max(sigma,self.sigma_min))
            k_rec = k_d.view(b,c,2,h,w).permute(0,1,3,4,2)
            # Image conv 
            img = ifft2c_torch(k_rec)
            img_conv = img.permute(0,1,4,2,3).reshape(b*c,2,h,w)
            img_conv = self.I_blocks[i](img_conv)
            img_conv = img_conv.view(b,c,2,h,w).permute(0,1,3,4,2)
            img = img + img_conv
            # Rician denoise 
            img_d = img.permute(0,1,4,2,3).reshape(b*c,2,h,w)
            img_d = self.R_denoisers[i](img_d,sigma=max(sigma,self.sigma_min))
            img = img_d.view(b,c,2,h,w).permute(0,1,3,4,2)
            k_rec = fft2c_torch(img)
            k_rec = apply_IDC(k_rec,k_us,mask)

        # Final K-block 
        b,c,h,w,_ = k_rec.shape
        k_conv = k_rec.permute(0,1,4,2,3).reshape(b*c,2,h,w)
        k_conv = self.K_blocks[self.iters](k_conv)
        k_conv = k_conv.view(b,c,2,h,w).permute(0,1,3,4,2)
        k_rec = k_rec + k_conv
        # Final image
        final_img = ifft2c_torch(k_rec)
        final_abs = complex_abs(final_img)
        final_rss = rss(final_abs,dim=1)
        return final_rss

# Training 
if all_files:
    mask_func = RandomMaskFunc(center_fractions=opt.center_fractions, accelerations=opt.accelerations)
    train_dataset = FastMRIDataset(all_files, mask_func)
    train_loader = DataLoader(train_dataset,batch_size=opt.batch_size,shuffle=False,num_workers=0)
    print(f"Successfully created dataset with {len(train_dataset)} slices.")

    net = KIKI(opt).to(opt.device)
    # Calculate and Print Total Number of Parameters 

    total_weight_params = 0
    total_bias_params = 0

    for name, param in net.named_parameters():
      if not param.requires_grad:
        continue # Skip parameters that are not trainable
    
      if 'weight' in name:
        total_weight_params += param.numel()
      elif 'bias' in name:
        total_bias_params += param.numel()

    total_params = total_weight_params + total_bias_params

    print("="*35)
    print("     Model Parameter Summary     ")
    print("="*35)
    print(f"Total Trainable Parameters: {total_params:,}")
    print(f"  - Weight Parameters:        {total_weight_params:,}")
    print(f"  - Bias Parameters:          {total_bias_params:,}")
    print("="*35)
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(net.parameters(),lr=opt.lr,betas=(opt.b1,opt.b2))

    for epoch in range(opt.epoch_start,opt.n_epochs):
        net.train()
        epoch_loss = 0
        for bidx,batch in enumerate(train_loader):
            k_us = batch['kspace_us'].to(opt.device)
            img_fs = batch['img_fs'].to(opt.device)
            mask = batch['mask'].to(opt.device)
            # Estimate sigma dynamically
            sigma_override = estimate_sigma_from_kspace(batch['kspace_fs']) if (opt.use_stat_priors and opt.estimate_sigma) else 0.01
            optimizer.zero_grad()
            img_rec = net(k_us,mask,sigma_override)
            loss = criterion(img_rec,img_fs)
            loss.backward()
            first_bias_grad = net.K_blocks[0][0].bias.grad
            print(f"\nGradient of first bias term at Batch {bidx+1}:")
            print(first_bias_grad)
            optimizer.step()
            epoch_loss += loss.item()
            if bidx%10==0:
                with torch.no_grad():
                    gt_np = img_fs[0].cpu().numpy()
                    rec_np = img_rec[0].cpu().numpy()
                    gt_norm = gt_np / (gt_np.max() + 1e-9)
                    rec_norm = rec_np / (rec_np.max() + 1e-9)
                    psnr = psnr_metric(gt_norm, rec_norm, data_range=1.0)
                    ssim = ssim_metric(gt_norm, rec_norm, data_range=1.0)
                    print(f"Epoch 5,  Loss: {loss.item():.6f}, PSNR: {psnr:.2f}, SSIM: {ssim:.4f}, Sigma: {sigma_override:.4f}")
                    fig,axes=plt.subplots(1,2,figsize=(10,5))
                    axes[0].imshow(gt_norm,cmap='gray'); axes[0].set_title("GT RSS"); axes[0].axis('off')
                    axes[1].imshow(rec_norm,cmap='gray'); axes[1].set_title(f"Reconstruction"); axes[1].axis('off')
                    plt.suptitle(f"Epoch 10 Loss: {loss.item():.6f} - PSNR: {psnr:.2f}  SSIM: {ssim:.4f}") 
                    plt.tight_layout(); plt.show(); plt.close(fig)
        avg_epoch_loss = epoch_loss/len(train_loader)
        print(f"--- End of Epoch {epoch+1}/{opt.n_epochs}, Average Loss: {avg_epoch_loss:.6f} ---")
        torch.save(net.state_dict(), f"SavedModels_KIKI/kikinet_epoch{epoch+1}.pth")

    print("Training complete.")
